# Deney 03 — TTS / Seslendirme (Notebook 02 çıktısına bağlı)

**Amaç:** Notebook 02'nin onaylanmış scriptinden seslendirme üretimi için aday TTS sağlayıcılarını **kalite (doğallık) + maliyet + hız** üzerinden karşılaştırmak.

**Çalıştırma sırası:** Notebook 01 → an onayı → Notebook 02 → script onayı → **bu notebook** → Notebook 06. Alternatif olarak `00_youtube_automation_master.ipynb` tek başına tüm fazları çalıştırabilir.

**Tek kaynak politikası:** Sağlayıcı çağrıları, model kayıt defteri (`TTS_MODEL_REGISTRY`), süre ölçümü ve `selected_voiceover.json` şeması `local_pipeline/tts.py` içindeki **kanonik** fonksiyonlardan gelir; master notebook da aynılarını çağırır.

**Sağlayıcı politikası:**
- **ElevenLabs `eleven_multilingual_v2`** — yayın kalitesi için birincil aday.
- **OpenAI `tts-1-hd`** — opsiyonel karşılaştırma adayı. **Kullanılabilirlik ile kalite farklı şeylerdir** — bu notebook sadece kullanılabilirliği takip eder; kalite kararı bölüm 5'teki elle puanlamaya bırakılır.
- **gTTS** — SADECE `development_placeholder`. Asla `production_ready` olamaz. Ağ hatası olursa `failed` olarak loglanır, notebook çökmez.

API anahtarlarını Colab 🔑 Secrets panelinden ekle: `ELEVENLABS_API_KEY`, `OPENAI_API_KEY`. Hiçbiri yoksa ücretsiz `gTTS` yine de çalışır (sadece geliştirme/zamanlama amaçlı).


## Google model router (bu notebook'taki tum Gemini cagrilari icin tek merkez)

Bu notebook artik `genai.Client(...)` kurmuyor ve sabit `model='gemini-...'` yazmiyor.
Her Google cagrisi `local_pipeline/google_ai.py` uzerinden gider:

- **Colab Secret adlari:** `GeminiKey` (eski ad `GOOGLE_API_KEY` hala calisir),
  `YoutubeKEY` (eski ad `YOUTUBE_DATA_API_KEY`). Deger hicbir zaman yazdirilmaz;
  yalnizca `configured (GeminiKey)` / `missing` durumu gosterilir.
- **Tek anahtar, cok model:** ayni `GeminiKey` projenin yetkili oldugu tum modellere erisir.
- **Kota PROJE bazlidir, anahtar bazli DEGILDIR** (ai.google.dev/gemini-api/docs/rate-limits);
  ikinci bir anahtar limit artirmaz. Hesap/proje dondurerek kota asmak yasaktir
  (`ALLOW_ACCOUNT_QUOTA_ROTATION = False`).
- **Rota:** ilk basarili cagri rotayi bitirir; hicbir zaman tum katalog denenmez.
- **Ucretsiz katman veri politikasi:** gonderilen icerik Google tarafindan urun
  gelistirmede kullanilabilir ve insan gozden gecirenler okuyabilir. Ham telifli
  video/ses varsayilan olarak YUKLENMEZ; yalnizca yerel olarak turetilmis
  transcript/skor ozetleri gonderilir.
- **Kota defteri** (`results/google_quota_usage.json`) YEREL TAHMINDIR; sunucunun
  yetkili kotasi degildir. Gunluk sayaclar Pasifik gece yarisinda sifirlanir.

Gemini TTS rotasi (`gemini-3.1-flash-tts-preview` -> `gemini-2.5-flash-preview-tts`) master notebook'ta `TTS_PROVIDER='gemini'` ile kullanilir. Her iki model de PREVIEW oldugu icin cikti HER ZAMAN `review_required` kalir ve yedek ses/model onceki seslendirmenin onayini ASLA devralmaz.


In [ ]:
!pip -q install elevenlabs openai gTTS pandas > /dev/null
print('Kurulum tamam.')

In [ ]:
import os, sys, time, json, hashlib, shutil, subprocess, zipfile
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd
from IPython.display import Audio, display as ipy_display

if 'google.colab' not in sys.modules:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        venv_site_packages = candidate / '.venv' / 'Lib' / 'site-packages'
        if venv_site_packages.is_dir():
            import site
            site.addsitedir(str(venv_site_packages))
            break

import imageio_ffmpeg
IS_COLAB = 'google.colab' in sys.modules or os.path.isdir('/content')
PROJECT_ROOT = Path.cwd()
RESULTS_DIR = Path('/content/results') if IS_COLAB else PROJECT_ROOT / 'results'
TTS_OUT_DIR = RESULTS_DIR / 'tts'
TTS_OUT_DIR.mkdir(parents=True, exist_ok=True)
FFPROBE_BIN = shutil.which('ffprobe')
FFMPEG_BIN = shutil.which('ffmpeg') or imageio_ffmpeg.get_ffmpeg_exe()

def get_secret(name):
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    return os.environ.get(name)

ELEVENLABS_API_KEY = get_secret('ELEVENLABS_API_KEY')
OPENAI_API_KEY = get_secret('OPENAI_API_KEY')
print('ElevenLabs:', 'var' if ELEVENLABS_API_KEY else 'yok')
print('OpenAI:', 'var' if OPENAI_API_KEY else 'yok')

STATUS_VALUES = {
    'experimental', 'development_placeholder', 'review_required',
    'approved', 'insufficient_data', 'failed', 'production_ready',
}

# NOT: audio_duration_sec() artik kanonik tts modulunde (tts.audio_duration_sec).

results = []
def log_result(task, candidate, quality_score, cost_usd, latency_sec, notes='', status='experimental'):
    assert status in STATUS_VALUES, f'Bilinmeyen status: {status}'
    results.append({
        'task': task, 'candidate': candidate, 'quality_score_1_5': quality_score,
        'cost_usd_estimate': round(cost_usd, 5), 'latency_sec': round(latency_sec, 2),
        'notes': notes, 'status': status,
    })
    print(f"[{task}] {candidate} -> latency={latency_sec:.2f}s cost=${cost_usd:.5f} status={status}")

## `local_pipeline.tts` modülünü yükle (kanonik tek kaynak)

In [ ]:
# @title Yerel pipeline yukleyici (otomatik - acmaniz gerekmez)
import sys, os, tempfile, subprocess, base64, hashlib, zlib
from pathlib import Path

PROJECT_REPO_URL = ''  # opsiyonel: gercek bir git repo URL'si verirsen modul(ler) oradan klonlanir

_LOCAL_PIPELINE_MODULE_NAMES = ['tts', 'approval']

# Her modul icin: (base64 kaynak, SHA-256 hash). Hash, gomulu payload'in
# bozulmadigini/kesilmedigini dogrulamak icin kullanilir -- eslesmezse acik
# bir hata verilir, asla sessizce bozuk kod calistirilmaz.
_LOCAL_PIPELINE_PAYLOADS = {
    'tts': {
        'sha256': '7df5642ff7d3b8594b5b8eaf3a9970a364b6bd4206eb43cacd645254fe56d94b',
        'encoding': 'zlib+b85',
        'data': (
            'c-qxlTT|mmvVQlks8K&)ov;Nxy|XbvhtCP7+p|k711!+JXM!MvY?W;-vgFYP+QZ?$pD(kjq|(I=eR(*C2e758tjw&;FE3R%j&mW)'
            'Jeb8&6!KFsF4A$nk#fEgak`A=B3}iW48?3K4&L}hk^A>)G?!kq*(Ng)B;nxTz~822QO=|vltsD@iYQI|b&wY_Q^UN$;504dEKTn~'
            'MBGWa$wgG;B7I22x{Qlxyvfr0D3qC)$z;A-2iaYeEX9Z8x1v}{Vcw4CK^zYrvS723Sx&3->s-ciUdT|fmfmCCf9Gi;=4n!7!MqT&'
            'GKz_K9NkN?k{PC?VwM)GL8AG}m7F3?q=?p=SgvJKuq4F11W8eD;xq`sbS5_CEROP(Bo?;C3aolU4ccT;QsnzkUA|8|MtMPl#L@XN'
            '<lcG%$8iRWEL{uVUqES@^nF?|%?b<yH@s#(80crKAYaAN%zP!W%;$Vn7ExS%D`(KsT;{oXQbcR1mI{MH(l@h^`99(?e?n$8ZW9!='
            'm>G7CuY<wm)zQ_5OaJ51$%pBsn25)Lz>gz8LA;2y6gwl~gb+VYH}t^Y#KByyQc^j+%H(|{AN)-IQARK&XQU`#6S1xIT{uyamy1O-'
            'k7zpB3+Uk@h+_2`1H*F8c9qE>+=9f@U~qMH>Hl{2etP0xO#gO#d3Eu-#a;}4k~o;<&Y^g8)@dkXKML_hzw+7cVr*R*#Qyz{v<#b%'
            '9`)nv^})f_eEpZJ`2PJLKSy`}I{M9d(yXT&nFLXXGQ4B_db|oNTJ#VWn{Jg5L>*f5U*Gw2SX=IwIn6up4&EFX(whh)I+Xe8I8I^z'
            '!~}#NgmNK*GK^9`EHmYExttFd3n&Y&IE#`IZ>THA|0xzQ-9r42IK^Hc@<P!<z}kd0Ts^0sOu{e|9?Z|ku%O!7#JsFbhSl)v#+uIf'
            'o}`somh$e8^HuubV_&mK=De+G28@^(u6azlpg&~Y_?s8mkU75@Rcp+H4Lb{fW>XfEtE?pJ1zxF-JkB+O5tp74Vh$yWA12~pw{%6e'
            '?Yzv=G6{#QwIOlkvFi_+a9)Jq!oAjlI5*c$CCiOF5^oeA^3z;y3NhtBc-PN>+DC#lzcyKd+2K4UXMi^5S+ps9lJ$^OGZI+cC;g0s'
            'F%kX3*e`$OeGj8~p%w=2+$xf{B4S9Y*Yng1Z^}<j@MJts*H|K=sF2S74RjSl;9eo~f~i3Rc;$+Rs91?xkC^j|)K`JS1MUI~i=p9X'
            'Qgb@dyXM{wYE=-_gP{lkgX2)TK)5NSn6Jv@F82sBX|RzSrby5<G~eCsHWmjMMBwJpuc2GLV2{B&E%9@Vn-7)%=7Sc*+Maq9Rttf|'
            'u-+@@Vdl6Z$i<>&lYU@DN>C9cioNE09^*Y^(9V#@)yu@d@npVjiYmsCzV@3mFYtaCgBP1su`ga992~gfKL9^(4%)W)=$MdOBPN3>'
            '<X{Ehajgg&I4g~Hgm4sEkv1H*(0T(wf*2({^kHHV+={T1p5c`ili&>x^|z;9CnNph*<n{6g-3^C;nY?RzguQe9*J3y2lU}j;4Y>='
            'IZw7@DzyJ5r_`UjA>dF!^Rv#Xc<|2f=jgb{JIUgB>aE9W_KZa%Uczvgr-^0;f1`v@)V;MG5TEl6at~<}1KToKlZz}6j1?^3epipA'
            'RcnXb&3tWcN(laKB<b}+qTCjsVRql!le*jcdB4(W#LIl@|E@zRqZCNTUg%Jq&9FbIPsker*b0fDa}*QHls=SS?e@8@_xzek&{&jl'
            'ycL`^F?aV-!1cm8tt#G~93yK_O=><uJ|q%ZJfztjvx&r?eZb~Xxg+$rUQ_Xr1W2lfu(LEqBFps8kFNgiQjHLB_9cpxmQ-mJFDpOE'
            'd09}Ldn<0+Qfm5&WiL&VPf8KWFa(*f$Fz>bI?8kMTg4cysK%%bVlHDD*gNHU-V^)<by*N1#;O1uL_}po1rZ5NWmR5g3ncJjualMU'
            'VLYTkBCrDGd4}SISQw9)R!-WeYOWyfOKbL9h>C(sP@a$6?uDqL6W502?6yuv?KM_KB^G5-9IS&dI&4fA+hBtiNl5S40Tk;z>Fh*p'
            'ERCc5mkTcMVHM3+4a=<<YfQjSk|}eQ%5pTOc=8j~hC|n8zOkG={BsL@);D*EuLYl#ahdQVF?_Vwdvdi3w6@!JO6Eq-S+4fWe%W5i'
            'T8^rkX+=%;+N)G7zAVf<Q^Rl_V{4bHlCgqiRgrY0<PoYK;n<?bRn~gbMC?wW8oj8fsSBIW6gOr&%-TI0@XRK(V4FbAtT1_OE&1dE'
            ')lDe`QrCw+9UR;|>Fs!7IPDFHs`$0y;ST!(gsqz)l7q%~h>)nozkh(-q2%7!xTLW7x%3i1wJd|BN-3dXryz~dG#{v4q}g^t%IL<#'
            '%cJ+xcT=&WQ3DVcOA3-4fEI|K#3;NF2nMt-#EKer0TcUMnl6!gKFFDnNw`U)L?xgUL91wrfD%;kR71S#y+yg!<c7O;`-mlBERpjh'
            '+(KI*(8Qj&wM7~#hauUKv_e@y8H1f7RUw*&dc^(hTdq+l^Z*|dPK_Z=a3_VPL;*Q~;e^z>!)77QY9<G&&w#-HJ=0rwtSv|vw7I^c'
            '?i&p=tvB8YSXN`Icm)Oj5y04ge*H!aVX<>+L6#ep<09S%dCoaWy*$yJh>j|oI7Dil7`t+bWl{sBC_{UuD`br7z<y+}<K@Bqw<OrT'
            'h(L?kfWQyN+5nvYM;O36YT;|J+|Olp4|9NX1%QPV3XkF_b8T371-JZeKcIGQ+U%-jPP|k9fs$g?u;;9UoZcI}!uWrd2+zfKyofSX'
            'KU(dF!YZNryQ!YgMS*FFlGDGULiX(Z>iF#R=mgm@d~g9fn+5YbPv_(n`0R5EJR&$515U;aNYx<$0TRezG<GeHInf_4fFVm{;dQeZ'
            'wx)oM&l)($(wx0kLwb*oF~A)+q{-|@qD<llph3IlRk*0|X%oOf(}g*hFl(XSLr%tvX<?ycjD_LE+wl^ALK}Nq=t_8XI{i4k5Hl%E'
            'XB<R^xisGaiaf$UIzPWS`*`%r$<(J0$M2^Xm#+7nZDjz`??1ew&GM<yUadqmaH}I!*Rd*qyMy=BUynYVTs3A=tg_|}FzBQI>(R-{'
            'FGuhG;a|SHI6lXO>3?6*3My#*Qlh9>B@$^4l5TMqt#g?`%QJaLCp2NSL^zGeno$77<u=Vcag+q&4r;%HHIERVklo*5N(e_UlQI#G'
            'K7Gic{4k(xxC@vFf?+~#7?w*66zMX<T1@g>-cd4&;R(W8l2KeEf1L++|0Ui@m`p^^4e1<ow#4%i8w>*CUMkj!MgV{R6RU-CiCV%L'
            'xH|ar5fz`W&8ZThBO2~Ev%xn@V>(fI@Qv#rnNL%AN^p-DOUCP)SvltblwMzB`c0JxoMm`P@ivs>u-wGaoQ<>I@Y>OoQ4>T^B|~mN'
            'P6Mpe#qqx=>NokB{2EY{C{}QosunGf(gQ%wPNx-Zogq;58B1MmF=_S%?zWg|v^K*9N9W2NsYaMd<zNCPMP60;9}(1vh2@7_jHL-a'
            'Jgi_ENI@WdRNy--7`h8Rqa^-Y>68wZHmrvYw`se&=G2_t?eZHHI?Pce)pth8pxWmzrv9p?P=^1R1+4IcqF#!RL0rly1)ye`vv4TR'
            'Bx|{a29T5??(xp%!7>iEP)Cj`?9r_HL-r(w>~D!o@SMzPxQhiBMVVE#sEmqDXd<p}Is)06thwh29v$%};n0k&O6->Lw`L)op;^{e'
            '?SwJ2U-cf7ZETE|nVuFwhjS%KhX<7v@yqMO*EbD+?w(Bhh#nDTPs7U34>?b-+`F6n>nC{;6lJcSb<&(2KV^=^5q2tWeMQWmhIMo|'
            '>Cc~b{QRLZ1CUEm!M^r?Qq%j(la~?=3KVvm0y2`BX^ntLe?jvLsW^Yx4FJ|Kr`bgDAxgOC4U5oOzupRJuppB`dS<{&U47{3WQqOu'
            '>44a;N(e6FTn&;}1ci$-QF+3<qtnx~E1fxjbKca+o{9N;8o~<HEIAcaCip8r%9bT%4+wL1?qQ&0FQ?=wUs=gc5I+Rlo}7ot4U=@1'
            'hFiL2z#5k%E(#>9zm<ji7WlkWpwC*V$noU89hD@fQw-uNU$PP!LUe|3<e}?SrD&>zv*cf~>*orlsik_$xu9QVrd2wm=hQCc-YQe$'
            '7M;+tDhs*%`_cHvpZ;nRH+2q3Z5#f&D%ao_Bw*E871C_mu^>;&43&oVMY@*(!_?O^ynGda_|Pug3DzCQ^(6O9hE7>5#y>l*>#gJ`'
            'BrXVaLl=1QR|Shm42+J<9fsy5i9^A0byPp{XIoSBSPz4uQwceA$tCX@OWr}7r?_v-dm0$Pt$o8ZG)XX3*bq?GI7TlXpL&{p4fl7H'
            '#ZxWUbL*pTDBby>n*oO;i9Tj9OU~0W4#~u{KZU4^bz@w2BJz=~T(v<XZqTD}Fn}Y}T@$y>fK`fW<(WcMe{Gkat#Y#qsZQ69s4VR~'
            '9JNPQYHNIN5#I&(n6~Rxi-LWbq!wy-nOwb5&Qnim*3UG!xlkv7*i%&u;ksLWY)xJQsVJk<_HHgTc}e;(G<TZ3vY5MGl&9*R#?ald'
            'x4tm~M}?hXWfU-y>znp8h6>+ckfVb4ur@Jkym6wO!vP&~8oTwDvKZFZ<%-Ef_d1N*=}EOSsr{yiy6dV{Yp8Y`n!zAmm(^A6`Bp=8'
            'UHS6uw$^Pc3V@pwiSkx0LHSM}U+sHdJNX&5JVNHkF2g8WOO=ntVf0oUpI#oF;`h7h$@I7B<+1o>`j3<2-_I_nCe9<MJ%PtYDDzBF'
            'mL|bgI8EBYHd7^H5Ju}T7>S+1Lr^S$GPnoR$znTDR>N4^5j&{W_->7}z@;*b>-wp6eo&!~HBLRU2U%FuGPLTAWt>f*vItwxZR#(@'
            'tz9#o!>@0{t~PExcQbB>ytaON573)>?G70c()Chl;$uCaiIc4b?3W$VeF)ll)>!E?u==f#`W0~cQYba&BUzd7rGYBhH>FIYe4_3y'
            'u%xH7v2MKi>bgl!$)nSG`@Pt8OF>sAW<dL#s)u^#m}W#BQopayV~98AtID%db+ZSA89GN)!^mOuf_=lf1=Vld0v?-0n^LVqAwPj#'
            '<iy;&F-;gcw&%PVTeVde9fnm0mjaEdR!|3oDVH$?Woiks|7Aco5LDB1R)(;0QzpHmR*xyzOnezrsn9NSP=;j4+p`D>-R0HA@w=;&'
            '-^JzhpC6{DSI0*u_=2*b(%WrsYYnABb$>ig(~Y>8UVXSYy)=@kj(1+p`K-7irrX*P7ZsU=A4nZT`St`VoCS5!tp+)hkRj4gS<Jmk'
            'EUR<Y*-qU;Yjw@qS}<KMbvY?10`iG^0ks`f4OAb^?JL`Us~Tm91EvAuKz3B5d@sH_Hn;CdaD$}R)OkZRW1&~yPO0c93j6AcN<`;#'
            '<#I)hmK-zMw}M<=Nnb<B>2#&uO_ivd#2po^N?_Dw6nCUczXkM~gF0&2o?9&T;T-gm@|BiKt!Be&RhoulrDbdhj7+r<vHZ4odmg|y'
            'u&in?rCD2uqto}hT$OJCc;dLI&m9^D7T6aX%+i&$sw+YoS8JcL$3EX7c&p@A19c)>4c=Azd*Vu;@!GwMx^JOa>-Xs}jOY<p;F=5d'
            'uM^a?o4ywY?xRJcm8!$Q^`6Qo_HO7(MBPu|dtEhffw6TEO5KSN!ywW1w-Uc95yiTM>k)9FxYUgxwkIaVIfL755a_^T9~R@QYrpf>'
            'clbTrz*~|c*B3tcu7;ngE!XL0NzT*ActDnzX$L>$Da~gpb+B3yj$5@)yM(PwzI`Ul8A3B(0~@=JZASVk!sqLgR`Fo<OD3HiZOCwA'
            'oh2*Qx-(4Dr1Kobk=tIalB8d6Do^eCzZFvoAPuofBvsiBX>+>D)kxM{cVe}526o$1b;+x{UY&KUZZ|_XO?Yhnt4k-v8Cn0W(#UP<'
            '+dB3s$2DJRr0zL=80m$$A+DD6BJfHOO|2pQQU!R*R&E+Mm0lJ*npYtzX3Pmdb*9+p^n=yjux@8lqh0aUIxN$BYA)heLs<>x7@*0a'
            'x`Sm6n=}tLyVv*UizWE&{=3q(J?*A5O?O=<z5gE4+fvti)TBFVI<|qSeVb`-y?L{vO%LX;>;FXpHGWW4lsDa3ZsQ43$v%Y{qx=G8'
            '=_$`C%DS%qc_kToI~BChL&lFJ^PW+{-p$k3>8tv-<6psEpvzo-iN=)cJg2Za@#+=GcH>>8&Tbpy@!msl?>r|Q%*3>n^%<OEvn#N{'
            'O9K&&D?<%avl_OFSWn+%_m7j_VWNuC)m0vXxhJ<X`;{&rM6{-h34GdoNU6u7%uIgS&UsV?f-!u}q-U47=k)wZVy`Gl-qyx6doMi!'
            't6wd4DcxwJm{2>Z(*?6R6Z7G@egCT6gw3mPU3r@O+>WZ<G^Z$LJB?YW?>Bm9n~v7u@Cy$Ktc9C14HMlRbgYB<py12trgO%9PCVh8'
            '>aBjKI^Q2Qwt4c{`1(Wk<h~W<9Q5NzT}o9K;OU-qlbYSi)`9YsI7>J^4**+~GH?+@8Fo(vq$?>?l~yO`-NwYA(I9EUQ~O42177&%'
            '=WZ}={PY-WqaV|_>p0jAOBRoZFSq39uQI+6ZN45BEXKbb7rrAVxP$Ma!NHgMzNHTv00v*D<<9MmFH-AwP^UZiU#3|4ss'
        ),
    },
    'approval': {
        'sha256': '119794218b882b74fe3deb51c7ea08faa1cc706fe6145e48bc4a40116a202d9f',
        'encoding': 'zlib+b85',
        'data': (
            'c-p;OTXWmSl78o}n1ehBtfH9O-Ad{dT{=~YL?_%RlGl=x^{P}P2o6bvAi%}N3a!$A-><u8fB^_f%4YZY!IprTo}RvZeVKth&$|ec'
            'G>O6>R=YGWPFFh7Sx`i2qSoa$NKS*@E=#}Q^(rXzu-E(Tr_bu{r^&7Q<>vC!)mV-Hb8>rk+dJ+2(>u4bw`E?axmJO?9e+6gcz!ol'
            'd7*czSO<k#V-cO?>iqrP_#+-^RpvUw<6H${Q3i3mSGm?nk6sxL^+E+%5iK!bs6GUF9wjR^%fl?%6;okaFV60=QqNSB6e*KkmPyEa'
            'qh))RQqJ#Ws@m!xkq4H%igNH8Y<F1Fvy?p$m1)6ND~K=d@m9i2GjCC$PEWDnWtxFfvKMKpFfHQ+Hugm;45VR0bw~5CMv@j7sjR?z'
            'B=7+d!n;6a<$MmAFe6aIt!&6qmrjao4<%x=IWcEtQ4bmJ_OpSS6-7=zVXZ}Pmg!`nvuRmGu{>SqIGVFqAoL&%HDq#qQ*i~UtjaCc'
            'qUqRoqS6Ed^%tG>^cKv5;+IH2PBZ=AWt8c~O!;Z1X3HRo>6dTSEK2fnxs1XHRHlod2=KHwTYyBI?uhhs7YCtUr!hF2^`YG?-KL_p'
            'RL!ggW|i6KKf<`oqc6nqT4#FDBi@*=$F&wUk%mZo<!%>8M&q|XouB^eci*dY$-nA9NZDbU6tq2TMRuYW5cdz0>&waYkLvuxhmSYE'
            'o?ofE8+CDWeK)?oJ7W4`|6|ByUkfkOVVW)SdJ|K#+%#VYU<K+SP8QKh!(9wtmW|$gPqykCLoM_l1;iPYizrn~NEg3+i56OZf!Jo@'
            'T8EpdUEJ@_K<X4Sg`{$^R<qC&0&bZ6Nv^^&%SaA}!GX$COt;1aUki-K40@oNl}SPm@9u5~3i_r;Y+3Y>C<|vnS)^M`580)o1Z+i%'
            '$^&<_)3nTcmAqgo1-S_%kz-RyfmvaYWqTqE4Z*t}gNzh(ofd0)NcvahE>43*K2W=I9!L3F=gbss1)C#pN_6p<W*eYN_9Y5|Yo6!z'
            'SPawYvMkC>Pp2x{?$V459iGV!l=ph(8K7wmKe505o~MaSr`;40Ai=z`zeBP3`)itL8Mg}xB4CGoz^}cYv92Gk#?zl+aJTAA`5r_O'
            'YKAxRi^Ll!PXtdteM!MR3@M{uiVV<Iy%PC*{a)`IKnzLO(osZ5P~~!bF}a=GTvwVO2n+!R7g10>9l?<Tddw&Wf?|_75n+SXVU&}D'
            'Mn$gUrSiAwLX*jqS%`nPDY<pd#+{NsB&!&Y)3Vr=1*icP$-;zVQqwY8_1WS;ukuKeIWHJ&;MBBfHKU!}dv<y6VKxLgak^!s0ks0k'
            'jJn|(>uT8vn_IL4)+OLRg#mJais2dp>=-kT({KY62Sot5m={>iVaW!S7h1$kvj}pS5XrB$Y!}goJ*f9$e0vKdnSQuBzn)t0Pk$bN'
            'hO+C8kkVFgCehL|(-2pd&VhoVa*$HZgPivOkWAw!+>eMeLgj=I8S#ghH&{b1fS(p6hrlVtpFOEt!6OR{HYzqqxkE_R6i9}>>+!GS'
            'k5k7u>4(YX_#+wUlea=Z@t%p}bF0E_uowKLh;Nr?EGC5lG$$X&@Y1{h6I)VDB=%^Ak`g6z5TFt8n3qv(unHS2lZvnbMAAUuO6Syx'
            'BcNFMih5*wh=Y=uE!JtSU3yUhv&<!Hz)DWcue=209@kO0R&(rN)5;gfU^C83#1a!q5g3?k76hcrNqy(nmx@vpti?{nu^1i}Edk5-'
            'o;4ir;obw$MZ_TFJ-ov{f0?iy88c@Of&g>%K8SPR*~jsJe1ecKrys}H(9{&c>T+@iQ)5u+Dg5yY9j-a`n;L5+vtYe1ynzg9<x}=?'
            'CCV4xkh@|p1)@+swXoC!9Nm1J{5ZKjznY$Zx}4nnMFs_0rtktSX6-P9wR#J^L~;_uC46ye1WbvQUqjm>3X8rv{RjUr60)8*Hj{GB'
            'ZT6(HN(B^mjIoM=3SopHn*32CejqJ*Iw_N_E>1@3o-81L7^st4i+KDr91frHP;#Sb<J~>pJv<MY;5m}J?XK7>Ll7{P7Z!SNuCG2*'
            'R#-4_iBSfwA382*82kaT(qJ=824-o&icEi@cQltmE#fD;M~7*Pn3OuTsSMDq=6_~ho+=%H9h(ZOWgM(3AK-Kg5JC2DsJ-^b-$oL^'
            '&i)~GJ}@{}yJ)|Iu><+eh_x<XWHHwiBtQ*o;~R?c@qCgC$BW8JvC+JYIgp)l?u9Lr+rM#l$g9>YZ(vT5fjbU3grbrKQtD`gXpNNr'
            'Wuq9eQQQMjAK*Xt52ERsb8@PxURvqG_hfTkpTfe^b6+0T6YLA76D=K5GSawV-&QW9#!<CX=4r_J`NDtEkNd)@Rn}CZ?wM*bTOpy#'
            'Rjp7O-yk5f9L#j**}y-gEagxzl7F-W)0ji)Rf05>Tn3;L20LSS*3zd9*8EB=uR&im5CU&;iR>WFqE(b&i=6XM^KxE9MXYnLnx^3s'
            '>NxcCIN_8KjN=kOql4>_%Srv1WkJ&#YUjwy#3c7I7TGZShCwg|(ffgK9#Q4NI3sbe$BX7vDg;TW{ezJMc8z|`&o}CVb9%`%v#oJu'
            '!Bjz7L<7oA3W})Ax<#76%~Bj09WppIVCB74Dld?by^?n8`K|^JW;T~0A-$}gmsFH-JX25A1dm60BO{smxuYEktA72Y?~FTC=IC$>'
            '&iD=k^#t(zj#utVO}8C2ol3?%n=C;p@F|tmKK(U9RLwXqz`s>m@V<;=<s;cb#T%0U_p3I-TL0lx?g34M?odj9x_f{6cg5BF(1NxR'
            's;MJ~vbfI+6Yh9kf2flXO!{6~EKmRL_4`9CAoTryljqR(@TOFV`sw`l1HPSJ2lZ^`ziY1YFV3%Tt|u4gSL)wyQ3djn6n6X^RVjka'
            'M(=a35%L{U>AY_;h`MQ*r&25-<e`?10PkOI$<Lj!sxl1X$6%kUJ*BI8^`5C2p=El7S{`_o6PH+7QNYVc97t;*JYs<&Z%T3UG0Lf%'
            '$fazisovKMK@={-qkLn^J~HS45|#3glK5oND9|9MD2`A0fm5rwAy!ne05*|;hl_H%%WG7JGzS=xQ`aEK!zem4$q5N@N+RSwB!1Vs'
            '06`<o@-yEXP!t(?sB@vo0C~e1^d%;Fp!*k0#<2E+bx=FVr+&j<F=iqHdubZ68F1F-N%(z?kY=Vp6JDG;5_1Gq04NjzH8+tgT$jm)'
            'zzK@Z{5aUo7s1Gk98#(3fB)Tg|MoqkHHU9B!AsZ}mOIj?&y*SxGw%-tWSehXs(=fBgZ!8MexN$)16Oo5$?xbR+J1%UmT*w{V+VhO'
            '0TrofozSM4R<WkS*XGyEfZ1HD@xw6*<g1pAWJ)i$S_Zb8VrZIDx%y*hx-CU%;h6DtOQ|s)84o9wkKE1tZF2V$J&rPJa)IC6%&sf$'
            '#%!m9&&|vCldG{<mYHX&mo}G6N`6c;z?maOR}fLV;#U-HdYom98K(7H7cWTJBJR)_B?R-hR@Yt^%Gbk{cz!_QM=Eg23Ahmf0B94e'
            'G#Me1z7haba9J<85I2aZWgz)M6G0GyNw+oxw>y2cL1(lPXCMC3n@0#)TfN7Auor;E2C6Ev5)~xT{9E>{{JGx55Nm>GZQ9R$n`B!i'
            'D4A@x6>APE$ZjR!FEy|FZZ6bH$Sl2q@g&MyFv31=`trf{57Lx2=R<;I6@^r~suN4~gxq6Qs(re7haxK2V`LBp8#RZOdWS@#Jk|(T'
            '8{l&mDIhsw!d4^y8a4+XY}76&X)31~{lB^;!$z_Q)Co~MLF5PxKeQVJTKMXQx_@v9eu<E$S5dr(vZD+eH0;h9>^3?q#q@6KO_`<;'
            'qeZ(=bKZFevT|PjX%R_>2Z^2#X=vSL2nt8`f@~nv#)aXw8>qAYu1oK2(*R&)(dJDC(3B_VjSeFnf^_=xZZUFKo$)mA$n<wC)H>Ka'
            '5X7ZcDgm%V%E0f)X-;`nDkfmfr#{Q2=EEI2lgK^O4>h=@0*7F=dVLO+nMx=R9*628%`$1cNhb+TPwjCU&dYHV-z69a1-JO^H2Y)6'
            'aZocA<BWoL&^9@KD&_k9Lp3%m_2E!@crel!rxB0AXd3yaZPYX)Ydtrtbc4!ctG@+hL8W<VqBWAxO`4SXHt4cYN_?Y2>4mY!!2z5F'
            'aj_4+MheAP@;2<+fdDdQ>P_4~c~r_hih#s#>4>R*1{lxep{^eyLVHf3)nu6uWRa9&_&iNx-_9|Qr<9^Q?Tdc@nRCEL?qHG{9=ZsV'
            ')`Z!xgW0f)+3<B{yZ8;?#&1aclA%1j5oC?nJD{OH$E%G_x;&ISAGX7)SBD_$Of!*|4j29Uaeo-)QWU%vsFpxo#6=X=8I^KrzDK};'
            '-Nt1SY=cFFWTPgFG{ie52VTwLz;q`mld~IqOLvkJz-c|L*#mOOxf`z^cW%66(<c4w+;2^--fzcVokdNpA!mhiO?Adpp!Ah;qj4Nz'
            '>y7I4l5fZ)S}*vrhg8Z{buqv*vRzw>bFZCA@gXW6?WO;%!*z-@0(FFwxT<Rg_-$F#SZtM3<+I5$rPIH<)k2$r1JOTD1IA9A(h)^T'
            'j@rr><Z-{*MRKl0V8%5vo<=vw#!9@CO1pl^MG?7CdltW-1gp_Qx__`uAgOltPU90hyl*o@&GpWzvDRR}MZsa0W>?1@Ko`1O<5VmG'
            'Uu9`?V0T25VJbd`9Jm>cd<j(KY_uVG&JXy40NKy+Bi_-=+hO^?z;QE50#Jfg9PAMh)yY$Xp%cr{N&opo?a~-J+61T-Gvu`pO&nBl'
            'ZyS7V1xn99a($@pguwoRfXo$%b}}XvCY_yeXNV4oQ1E;nUoz^O`rU6@1|7lebEM+MLCRygK3aJb^~mR;bnnYECpvsuKd0<qM-Ud6'
            '$M&pJR&(6r8{iS&PGbKhl&jCMsG_$0Y0f1d*K{way1h`HxJS7|uRi&%Gc7tIh4x-NIc-E9dUJ0Y!w;?Q^XqogvYnQj(s*w6HWuk0'
            'o84)7Y$fmM>_y#NpMO^4>mSEgR}*R{vPmXTH(zFh);DQh#xV-8QAa+u^qx)z>SXwP8YRMzIhB3rKM&PM2^~ZRMw@VkMrj@@xgesd'
            'ecPSlZ4wk6$i4R^O26nd*dY6_*iZ#behVHKbp8WUaB1k3+5+9xUIP_4`WOvQt?pXK-vi}@B75JgbvI4k9!a9A`)u>Qp#sZ922DTH'
            '#TM@{qRFj3O_$*eFEa{45)jT#EX5PPM+uYXgYHNgQ?nyH&Xl=u@XBpA+Y)RZ*av{Y-<V0|(|o=MVh_3`8ZP?7j#d-3x0AgQ{j$u1'
            'SU|uWYSj1t+?zQH3A$Wm;@bo&FGK19EX$ZGe8U>S5r^#R+9lZ8!V;*9yq8wCZ1xd=r{((R$i4B%FhLhinO>ZxbSiCf2m@u3YbjqY'
            '59~6&L0aEK?2}s3dBK--ilI6u9mPTqoUf1o=XBC-u4iTXcP`Y;4&Gx38Q0l~V8{tm?!E%Vyao&Ld%hy*TwY*mey(3S;DmPGes8Gf'
            'G(=egN3ISzZA{)C6ik<?$|8$a>q4#q1d?4fFBbi&D@Fgzc40Gbc769aUC$Y&zNT53#GoiZx1qKJp%MX<+64Or1@3T*f_K<l>ln85'
            '-d1IMN9J3GhUz8>$#Fx??YwmvA~Hs(is21#w+PB^rw*$$FBnggs*%6YE4sN#segvBpio^43Js(`giT^Vs5fBYkX*JXS7lNrEQqMt'
            'ku#OgH|oON5BXvI{sxXvg=jv(mo$THNE#^Os|+lBkej!X`fubU);*|al&UVo#u@;SJFsx#oH>**%965@;p+*~|7BkAhK34<YIyqP'
            '_;PY}IT_iL0Wmx3t8jw2iK8l7R^faRk<PjTxEt{8G&n~im?M^V1ZorBC9rc(Hma|YBFeMcyQBHh-eX66w~4Mq_^zq9(%_7~BD~Nf'
            '@j6{>^+u7M@6#;jgVDoW((kJcZ8<qe9pqAtL`tr1-can2@nsSa#l_i*>txTG)Dg!W(jocI?+@ejQD<Q7Z)Xh-?DMw*#PZN&1PoZV'
            'YR(;U_I1<frA1C6<v$%^@P70!-@_P&!+uK*OV5HUjPUl;`Ss5e3l4h(V*knAhC_uKD*2WaG*lDeK{#AQU2u}8Nd^GY2|`(7dwVda'
            'G#2Z4*9CU4zzm*XF^^3V6zpaX)ZhPquO0ui*8WPJ*b<>zHXf@JNuC*edIOQQipHLs-@+sKUP}|}?Av?W&Y=3Ej`5vi=>Bz1SJp9f'
            '>B&iPwMww0F1bn15O1HH;QQRB$8)FkUEyY1rs876t%t$k8_@JdZWhorCZN8n{g`2xq>sM+o?%!PA$8VLpaZgR-)-e^5rB#OFC|&6'
            'rp%GgV5oEJc#qqnF{MqF!<mEp=9ex<i<_>#n*Ac?pk!#23i7#Ax<x5ZP**7PMVY)cteG2{K~ZP?EUxyw;HtJkA<1VR^n=*L$+LBu'
            '>KisLm7DmFU6|`=)?$3;0J=U%>9=XgJrsV;KMSpjThs4=xJ<IpESp}UX1D}3D>!rEM&*Q1)f(@J@Y`<0)}I08>h|vZYRpP-`yPBv'
            'maO1-rBRg_brDwIHKZe}^Q+q%y1ry{ngyj?;$rou8)}vE#}IN!h_4pVRRQ<%gv+g*a>0MO+B!XzhTj*ykdT(Ewev}oeAI>Cx4DkV'
            '3`Ma<c+ILn?ice%7miz-%SLi{*ZJK5%m~w~TfE)PSBkn0MW?$+>)m?|sH$oT#??s9ol%%c-QEIgd3w@E(b4zXT}vU~>$yu+Tr_&6'
            'm1`ZG6STmud0ub~URouI(0*~-tuo}WNrC91jw~m2Ky~KFEv8n8lEgu)xyelVm=6^c3V`S<V&NV^KcHf&ck~$A6=tK4ZGHHSjqBIj'
            '(w$b3mqD_LT3OXv=fO@wFCU(O%=3#@K$ne$T{j`L8tO;}U(=MyEPfncARU?f*T^zRN?5?ZCA}gO&bJ3i`qJ}l?vR{4SuUzv4miP^'
            'KC`J_o#hSHXOpp+>=R|DA@yhU_>KD2!H6=khzd@xZ|+q68Lep{3GR)WxW}y(LyobtYHHW%24J=RioyA4z!^{1_5axg;%nNiYs6Z?'
            'f~ulXEtzr-4F_)N=(JXer_D=MB4z6BF7ffA@z{a;E2t;X-GaP>&2uKJTu(EQ5mAQEa1lq1>&130PLeu{!FH<jV)M6tYumgp1oBrK'
            'z#q-W(1r79srv2w?&7EMrCf^?R^__O&8Iu(R!dY=4N|^di8?2@of2+p7@{VHmM_J{a|rX}6n0%*5VWo$^{@RHQOw>s98nX@$><_7'
            'b!+XR&a5HNnYooo-`2}rI_GP8{ydZ%m-{VhCQ)QYA#2u*WBh?pP@NXZ64GJhj};~5__l6LQX=Di5y^*92cO~}?RUSkHM|4+QS<9_'
            '>F-uwxN@JpO6A1q3=GEf?I+*BKImyST7WON8=YZwyV2IVRnKux_mhtG9A9f?SoCU#z?XZKsyYMwCv-C=*B9ff@h@Zb^ZDn?Nuxq`'
            'dKtVy|A|hAk8sw1p^sqzMRa(yZ$Sr%PVeHNLaywjvqH9S@4be*w2M_c?66pM%)uMEHq<rde^;y?5nX37`dW4SYyz&hIQ9K8-yUkT'
            '8m<*qoB!rP(3&m^O#vfQu5HZ~`eq?pl)G5p3vScj4;8dtqzQ@*WCo!gF(q<#{=%c``qrOGQ1llb<W$(N|1W@6T}_72my@kU`k0sm'
            'V+XNlKH8GA5iX78<0&~}q0=JgKM=_!!>F#39DXO@4XQ6u&g>@;T^|lqn=#*I&`y0B)Hnz|x}o|7Nw(knA1&#y3j'
        ),
    },
}

def _load_local_pipeline_modules():
    # 1) Zaten import edilebiliyorsa (yerel .venv, repo Colab'da klonlanmis) dogrudan kullan.
    try:
        import local_pipeline
        modules = {}
        for _name in _LOCAL_PIPELINE_MODULE_NAMES:
            modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
        print('local_pipeline modulleri dogrudan import edildi:', list(modules))
        return modules
    except Exception:
        pass

    # 2) Drive'da veya proje kokunun bir ustunde daha once birakilmis bir kopya var mi?
    search_roots = []
    try:
        search_roots.append(Path(DRIVE_PROJECT_ROOT))
    except NameError:
        pass
    search_roots += [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for root in search_roots:
        candidate_dir = Path(root) / 'local_pipeline'
        if all((candidate_dir / f'{_n}.py').is_file() for _n in _LOCAL_PIPELINE_MODULE_NAMES):
            sys.path.insert(0, str(candidate_dir.parent))
            try:
                modules = {}
                for _name in _LOCAL_PIPELINE_MODULE_NAMES:
                    modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
                print('local_pipeline modulleri onceden birakilmis kopyadan yuklendi:', candidate_dir)
                return modules
            except Exception:
                pass

    # 3) PROJECT_REPO_URL verilmisse git clone dene.
    if PROJECT_REPO_URL:
        clone_dir = Path('/content/youtube_automation_repo') if os.path.isdir('/content') else Path.cwd() / '_youtube_automation_repo'
        if not (clone_dir / 'local_pipeline' / (_LOCAL_PIPELINE_MODULE_NAMES[0] + '.py')).is_file():
            subprocess.run(['git', 'clone', '--depth', '1', PROJECT_REPO_URL, str(clone_dir)], check=False)
        if all((clone_dir / 'local_pipeline' / f'{_n}.py').is_file() for _n in _LOCAL_PIPELINE_MODULE_NAMES):
            sys.path.insert(0, str(clone_dir))
            try:
                modules = {}
                for _name in _LOCAL_PIPELINE_MODULE_NAMES:
                    modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
                print('local_pipeline modulleri PROJECT_REPO_URL klonundan yuklendi:', clone_dir)
                return modules
            except Exception:
                pass

    # 4) Self-contained fallback: bu hucreye gomulu, build sirasinda
    # local_pipeline/<modul>.py'den birebir uretilmis base64 kopyalari, SHA-256
    # hash'lerini dogrulayarak diske yazip import et.
    fallback_dir = Path(tempfile.mkdtemp(prefix='youtube_automation_bootstrap_'))
    pkg_dir = fallback_dir / 'local_pipeline'
    pkg_dir.mkdir(parents=True, exist_ok=True)
    (pkg_dir / '__init__.py').write_text('', encoding='utf-8')
    for _name, _entry in _LOCAL_PIPELINE_PAYLOADS.items():
        if _entry.get('encoding') != 'zlib+b85':
            raise RuntimeError(f"Desteklenmeyen gomulu payload formati: {_entry.get('encoding')!r}")
        _source_bytes = zlib.decompress(base64.b85decode(_entry['data']))
        _actual_digest = hashlib.sha256(_source_bytes).hexdigest()
        if _actual_digest != _entry['sha256']:
            raise RuntimeError(
                f"Gomulu local_pipeline.{_name} payloadi bozuk/tutarsiz: "
                f"beklenen sha256={_entry['sha256']!r}, hesaplanan={_actual_digest!r}. "
                "Bu notebook yeniden uretilmeli (nb_builder ile)."
            )
        (pkg_dir / f'{_name}.py').write_text(_source_bytes.decode('utf-8'), encoding='utf-8')
    sys.path.insert(0, str(fallback_dir))
    modules = {}
    for _name in _LOCAL_PIPELINE_MODULE_NAMES:
        modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
    print('local_pipeline modulleri: repo/Drive/PROJECT_REPO_URL bulunamadi; gomulu (self-contained, hash dogrulanmis) kopyalar kullanildi.')
    return modules

_local_pipeline_modules = _load_local_pipeline_modules()
tts = _local_pipeline_modules['tts']
approval = _local_pipeline_modules['approval']



In [ ]:
# Kanonik TTS model kayit defteri (local_pipeline/tts.py). Resmi katalog
# degisirse tek yerde -- kanonik modulde -- guncellenir, notebooklarda kopyasi yoktur.
TTS_MODEL_REGISTRY = tts.TTS_MODEL_REGISTRY
print('TTS_MODEL_REGISTRY:', json.dumps(TTS_MODEL_REGISTRY, indent=2))


import sys, os, tempfile, subprocess, base64, hashlib
from pathlib import Path

PROJECT_REPO_URL = ''  # opsiyonel: gercek bir git repo URL'si verirsen modul(ler) oradan klonlanir

_LOCAL_PIPELINE_MODULE_NAMES = ['tts']

# Her modul icin: (base64 kaynak, SHA-256 hash). Hash, gomulu payload'in
# bozulmadigini/kesilmedigini dogrulamak icin kullanilir -- eslesmezse acik
# bir hata verilir, asla sessizce bozuk kod calistirilmaz.
_LOCAL_PIPELINE_PAYLOADS = {
    'tts': {
        'sha256': 'b38d96df5191f28bfe0f0c52e1b9daaed37de0eff822f036945f750e9e4bfad2',
        'b64': (
            'IiIiUmV1c2FibGUgdGV4dC10by1zcGVlY2ggbG9naWMgc2hhcmVkIGJ5IDAzX3R0c192b2ljZS5pcHluYiBhbmQKMDBfeW91dHVi'
            'ZV9hdXRvbWF0aW9uX21hc3Rlci5pcHluYi4KCk5vdGVib29rIDAzIGtlZXBzIGl0cyBvd24gbXVsdGktcHJvdmlkZXIgYmVuY2ht'
            'YXJraW5nIFVJOyB0aGUgcHJvdmlkZXItY2FsbAp3cmFwcGVycyBhbmQgdGhlIHNlbGVjdGVkX3ZvaWNlb3Zlci5qc29uIGNvbnRy'
            'YWN0IGJ1aWxkZXIgbGl2ZSBoZXJlIHNvIGJvdGgKbm90ZWJvb2tzIHNoYXJlIG9uZSBpbXBsZW1lbnRhdGlvbi4KCk5ldmVyIHVw'
            'bG9hZHMgb3IgcHVibGlzaGVzIGFueXRoaW5nLiBOZXZlciBwcmludHMvbG9ncy9wZXJzaXN0cyBhbiBBUEkga2V5LgoiIiIKZnJv'
            'bSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGpzb24KaW1wb3J0IHN1YnByb2Nl'
            'c3MKaW1wb3J0IHRpbWUKZnJvbSBkYXRldGltZSBpbXBvcnQgZGF0ZXRpbWUsIHRpbWV6b25lCmZyb20gcGF0aGxpYiBpbXBvcnQg'
            'UGF0aAoKU1RBVFVTX1ZBTFVFUyA9IHsKICAgICJleHBlcmltZW50YWwiLCAiZGV2ZWxvcG1lbnRfcGxhY2Vob2xkZXIiLCAicmV2'
            'aWV3X3JlcXVpcmVkIiwKICAgICJhcHByb3ZlZCIsICJpbnN1ZmZpY2llbnRfZGF0YSIsICJmYWlsZWQiLCAicHJvZHVjdGlvbl9y'
            'ZWFkeSIsCn0KClRUU19NT0RFTF9SRUdJU1RSWSA9IHsKICAgICJlbGV2ZW5sYWJzIjogeyJtb2RlbF9pZCI6ICJlbGV2ZW5fbXVs'
            'dGlsaW5ndWFsX3YyIiwgInZvaWNlX2lkIjogIjIxbTAwVGNtNFRsdkRxOGlrV0FNIn0sCiAgICAib3BlbmFpIjogeyJtb2RlbF9p'
            'ZCI6ICJ0dHMtMS1oZCIsICJ2b2ljZSI6ICJhbGxveSIsICJwcmljZV9wZXJfMWtfY2hhcnNfdXNkIjogMC4wMzB9LAp9CgoKZGVm'
            'IGF1ZGlvX2R1cmF0aW9uX3NlYyhmZnByb2JlX2JpbiwgcGF0aCkgLT4gZmxvYXQgfCBOb25lOgogICAgaWYgbm90IGZmcHJvYmVf'
            'YmluOgogICAgICAgIHJldHVybiBOb25lCiAgICBwcm9jID0gc3VicHJvY2Vzcy5ydW4oCiAgICAgICAgW2ZmcHJvYmVfYmluLCAi'
            'LXYiLCAiZXJyb3IiLCAiLXNob3dfZW50cmllcyIsICJmb3JtYXQ9ZHVyYXRpb24iLCAiLW9mIiwgImpzb24iLCBzdHIocGF0aCld'
            'LAogICAgICAgIGNhcHR1cmVfb3V0cHV0PVRydWUsIHRleHQ9VHJ1ZSwKICAgICkKICAgIGlmIHByb2MucmV0dXJuY29kZSAhPSAw'
            'OgogICAgICAgIHJldHVybiBOb25lCiAgICB0cnk6CiAgICAgICAgcmV0dXJuIHJvdW5kKGZsb2F0KGpzb24ubG9hZHMocHJvYy5z'
            'dGRvdXQpWyJmb3JtYXQiXVsiZHVyYXRpb24iXSksIDMpCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIHJldHVybiBOb25l'
            'CgoKZGVmIGNhbGxfZWxldmVubGFic190dHMoY2xpZW50LCBzY3JpcHRfdGV4dDogc3RyLCBvdXRfcGF0aCwgbW9kZWxfaWQ6IHN0'
            'ciwgdm9pY2VfaWQ6IHN0cikgLT4gZGljdDoKICAgICIiImBjbGllbnRgIGlzIGFuIGVsZXZlbmxhYnMuY2xpZW50LkVsZXZlbkxh'
            'YnMtY29tcGF0aWJsZSBvYmplY3QgKG9yIHRlc3QKICAgIGRvdWJsZSkgd2l0aCBgLnRleHRfdG9fc3BlZWNoLmNvbnZlcnQodm9p'
            'Y2VfaWQ9LCBtb2RlbF9pZD0sIHRleHQ9KWAKICAgIHJldHVybmluZyBhbiBpdGVyYWJsZSBvZiBieXRlIGNodW5rcy4iIiIKICAg'
            'IG91dF9wYXRoID0gUGF0aChvdXRfcGF0aCkKICAgIHRyeToKICAgICAgICB0MCA9IHRpbWUudGltZSgpCiAgICAgICAgYXVkaW8g'
            'PSBjbGllbnQudGV4dF90b19zcGVlY2guY29udmVydCh2b2ljZV9pZD12b2ljZV9pZCwgbW9kZWxfaWQ9bW9kZWxfaWQsIHRleHQ9'
            'c2NyaXB0X3RleHQpCiAgICAgICAgd2l0aCBvcGVuKG91dF9wYXRoLCAid2IiKSBhcyBmOgogICAgICAgICAgICBmb3IgY2h1bmsg'
            'aW4gYXVkaW86CiAgICAgICAgICAgICAgICBmLndyaXRlKGNodW5rKQogICAgICAgIGxhdGVuY3kgPSB0aW1lLnRpbWUoKSAtIHQw'
            'CiAgICAgICAgY29zdCA9IChsZW4oc2NyaXB0X3RleHQpIC8gMTAwMCkgKiAwLjMwCiAgICAgICAgcmV0dXJuIHsicHJvdmlkZXIi'
            'OiAiZWxldmVubGFicyIsICJtb2RlbCI6IG1vZGVsX2lkLCAidm9pY2VfaWQiOiB2b2ljZV9pZCwKICAgICAgICAgICAgICAgICJh'
            'dWRpb19wYXRoIjogc3RyKG91dF9wYXRoKSwgImNvc3RfdXNkX2VzdGltYXRlIjogY29zdCwgImxhdGVuY3lfc2VjIjogbGF0ZW5j'
            'eX0KICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXhjOgogICAgICAgIHJldHVybiB7ImVycm9yIjogZiJFbGV2ZW5MYWJzIGNhZ3Jp'
            'c2kgYmFzYXJpc2l6OiB7ZXhjfSJ9CgoKZGVmIGNhbGxfb3BlbmFpX3R0cyhjbGllbnQsIHNjcmlwdF90ZXh0OiBzdHIsIG91dF9w'
            'YXRoLCBtb2RlbF9pZDogc3RyLCB2b2ljZTogc3RyLCBwcmljZV9wZXJfMWtfY2hhcnM6IGZsb2F0KSAtPiBkaWN0OgogICAgIiIi'
            'YGNsaWVudGAgaXMgYW4gb3BlbmFpLk9wZW5BSS1jb21wYXRpYmxlIG9iamVjdCB3aXRoCiAgICBgLmF1ZGlvLnNwZWVjaC5jcmVh'
            'dGUobW9kZWw9LCB2b2ljZT0sIGlucHV0PSlgIHJldHVybmluZyBhIHJlc3BvbnNlIHdpdGgKICAgIGAuc3RyZWFtX3RvX2ZpbGUo'
            'cGF0aClgLiIiIgogICAgb3V0X3BhdGggPSBQYXRoKG91dF9wYXRoKQogICAgdHJ5OgogICAgICAgIHQwID0gdGltZS50aW1lKCkK'
            'ICAgICAgICByZXNwID0gY2xpZW50LmF1ZGlvLnNwZWVjaC5jcmVhdGUobW9kZWw9bW9kZWxfaWQsIHZvaWNlPXZvaWNlLCBpbnB1'
            'dD1zY3JpcHRfdGV4dCkKICAgICAgICByZXNwLnN0cmVhbV90b19maWxlKHN0cihvdXRfcGF0aCkpCiAgICAgICAgbGF0ZW5jeSA9'
            'IHRpbWUudGltZSgpIC0gdDAKICAgICAgICBjb3N0ID0gKGxlbihzY3JpcHRfdGV4dCkgLyAxMDAwKSAqIHByaWNlX3Blcl8xa19j'
            'aGFycwogICAgICAgIHJldHVybiB7InByb3ZpZGVyIjogIm9wZW5haSIsICJtb2RlbCI6IG1vZGVsX2lkLCAidm9pY2VfaWQiOiB2'
            'b2ljZSwKICAgICAgICAgICAgICAgICJhdWRpb19wYXRoIjogc3RyKG91dF9wYXRoKSwgImNvc3RfdXNkX2VzdGltYXRlIjogY29z'
            'dCwgImxhdGVuY3lfc2VjIjogbGF0ZW5jeX0KICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXhjOgogICAgICAgIHJldHVybiB7ImVy'
            'cm9yIjogZiJPcGVuQUkgVFRTIGNhZ3Jpc2kgYmFzYXJpc2l6OiB7ZXhjfSJ9CgoKZGVmIGNhbGxfZ3R0cyhzY3JpcHRfdGV4dDog'
            'c3RyLCBvdXRfcGF0aCwgbGFuZ3VhZ2U6IHN0ciA9ICJ0ciIsIGd0dHNfZmFjdG9yeT1Ob25lKSAtPiBkaWN0OgogICAgIiIiU0FE'
            'RUNFIGRldmVsb3BtZW50X3BsYWNlaG9sZGVyIC0tIGdUVFMgaXMgYSBmcmVlIGJhc2VsaW5lIHRoYXQgaGl0cyBhCiAgICBHb29n'
            'bGUgd2ViIGVuZHBvaW50IHdpdGggbm8gQVBJIGtleSwgc28gYSBuZXR3b3JrIGZhaWx1cmUgaGVyZSBpcwogICAgZXhwZWN0ZWQv'
            'aGFuZGxlZCwgbmV2ZXIgZmF0YWwuIGBndHRzX2ZhY3RvcnkodGV4dCwgbGFuZylgIGRlZmF1bHRzIHRvCiAgICBgZ3R0cy5nVFRT'
            'YDsgdGVzdHMgY2FuIGluamVjdCBhIGZha2UgZmFjdG9yeSB0byBhdm9pZCBhbnkgcmVhbCBuZXR3b3JrCiAgICBjYWxsLiIiIgog'
            'ICAgb3V0X3BhdGggPSBQYXRoKG91dF9wYXRoKQogICAgdHJ5OgogICAgICAgIHQwID0gdGltZS50aW1lKCkKICAgICAgICBpZiBn'
            'dHRzX2ZhY3RvcnkgaXMgTm9uZToKICAgICAgICAgICAgZnJvbSBndHRzIGltcG9ydCBnVFRTIGFzIGd0dHNfZmFjdG9yeSAgIyBu'
            'b3FhOiBOODEzIChtYXRjaGVzIHVwc3RyZWFtIGNsYXNzIG5hbWUpCiAgICAgICAgdHRzID0gZ3R0c19mYWN0b3J5KHRleHQ9c2Ny'
            'aXB0X3RleHQsIGxhbmc9bGFuZ3VhZ2UpIGlmIGNhbGxhYmxlKGd0dHNfZmFjdG9yeSkgZWxzZSBndHRzX2ZhY3RvcnkKICAgICAg'
            'ICB0dHMuc2F2ZShzdHIob3V0X3BhdGgpKQogICAgICAgIGxhdGVuY3kgPSB0aW1lLnRpbWUoKSAtIHQwCiAgICAgICAgcmV0dXJu'
            'IHsicHJvdmlkZXIiOiAiZ3R0cyIsICJtb2RlbCI6ICJndHRzLWRlZmF1bHQiLCAidm9pY2VfaWQiOiBOb25lLAogICAgICAgICAg'
            'ICAgICAgImF1ZGlvX3BhdGgiOiBzdHIob3V0X3BhdGgpLCAiY29zdF91c2RfZXN0aW1hdGUiOiAwLjAsICJsYXRlbmN5X3NlYyI6'
            'IGxhdGVuY3l9CiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGV4YzoKICAgICAgICByZXR1cm4geyJlcnJvciI6IGYiZ1RUUyBiYXNh'
            'cmlzaXogKGFnL3NlcnZpcyBoYXRhc2kgb2xhYmlsaXIpOiB7ZXhjfSJ9CgoKZGVmIGJ1aWxkX3NlbGVjdGVkX3ZvaWNlb3Zlcl9h'
            'cnRpZmFjdChjYW5kaWRhdGU6IGRpY3QgfCBOb25lLCBzY3JpcHRfdGV4dDogc3RyLCBzY3JpcHRfc3RhdHVzOiBzdHIsCiAgICAg'
            'ICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHVzZWRfZGV2X2ZhbGxiYWNrX3NjcmlwdDogYm9vbCwgZmZwcm9iZV9i'
            'aW49Tm9uZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgdm9pY2VvdmVyX2FwcHJvdmVkOiBib29sID0g'
            'RmFsc2UsIGFwcHJvdmFsX25vdGVzOiBzdHIgPSAiIiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgcXVh'
            'bGl0eV9zY29yZV8xXzU9Tm9uZSwgbGFuZ3VhZ2U6IHN0ciA9ICJ0ciIpIC0+IGRpY3Q6CiAgICAiIiJCdWlsZHMgdGhlIHNlbGVj'
            'dGVkX3ZvaWNlb3Zlci5qc29uIGNvbnRyYWN0IGRpY3QgZnJvbSBvbmUgd2lubmluZyBUVFMKICAgIGNhbmRpZGF0ZSByZXN1bHQg'
            'KGFzIHJldHVybmVkIGJ5IGNhbGxfZWxldmVubGFic190dHMvY2FsbF9vcGVuYWlfdHRzLwogICAgY2FsbF9ndHRzKS4gZ1RUUyBj'
            'YW5kaWRhdGVzIGNhbiBORVZFUiBiZSAnYXBwcm92ZWQnLydwcm9kdWN0aW9uX3JlYWR5JyBieQogICAgcG9saWN5LCByZWdhcmRs'
            'ZXNzIG9mIHZvaWNlb3Zlcl9hcHByb3ZlZC4iIiIKICAgIHNvdXJjZV9zY3JpcHRfaGFzaCA9IGhhc2hsaWIuc2hhMjU2KChzY3Jp'
            'cHRfdGV4dCBvciAiIikuZW5jb2RlKCJ1dGYtOCIpKS5oZXhkaWdlc3QoKVs6MTZdCgogICAgaWYgY2FuZGlkYXRlIGlzIE5vbmUg'
            'b3IgImVycm9yIiBpbiAoY2FuZGlkYXRlIG9yIHt9KToKICAgICAgICByZXR1cm4gewogICAgICAgICAgICAic3RhdHVzIjogImlu'
            'c3VmZmljaWVudF9kYXRhIiwKICAgICAgICAgICAgInN0YXR1c19yZWFzb24iOiAibm8gVFRTIGNhbmRpZGF0ZSBjb3VsZCBiZSBw'
            'cm9kdWNlZCAobWlzc2luZyBBUEkga2V5cyBvciBhbGwgYXR0ZW1wdHMgZmFpbGVkKSIsCiAgICAgICAgICAgICJwcm92aWRlciI6'
            'IE5vbmUsICJtb2RlbCI6IE5vbmUsICJ2b2ljZV9pZCI6IE5vbmUsICJhdWRpb19wYXRoIjogTm9uZSwKICAgICAgICAgICAgInNv'
            'dXJjZV9zY3JpcHRfaGFzaCI6IHNvdXJjZV9zY3JpcHRfaGFzaCwgInNvdXJjZV9zY3JpcHRfc3RhdHVzIjogc2NyaXB0X3N0YXR1'
            'cywKICAgICAgICAgICAgInVzZWRfZGV2X2ZhbGxiYWNrX3NjcmlwdCI6IHVzZWRfZGV2X2ZhbGxiYWNrX3NjcmlwdCwKICAgICAg'
            'ICAgICAgImR1cmF0aW9uX3NlYyI6IE5vbmUsICJjb3N0X3VzZF9lc3RpbWF0ZSI6IDAuMCwgImNvc3RfdXNkX2FjdHVhbCI6IE5v'
            'bmUsICJsYXRlbmN5X3NlYyI6IDAuMCwKICAgICAgICAgICAgInF1YWxpdHlfc2NvcmVfMV81IjogTm9uZSwgImxhbmd1YWdlIjog'
            'bGFuZ3VhZ2UsCiAgICAgICAgICAgICJnZW5lcmF0ZWRfYXQiOiBkYXRldGltZS5ub3codGltZXpvbmUudXRjKS5pc29mb3JtYXQo'
            'KSwKICAgICAgICAgICAgImh1bWFuX2FwcHJvdmFsIjogeyJhcHByb3ZlZCI6IEZhbHNlLCAibm90ZXMiOiBOb25lfSwKICAgICAg'
            'ICB9CgogICAgaXNfZ3R0cyA9IGNhbmRpZGF0ZS5nZXQoInByb3ZpZGVyIikgPT0gImd0dHMiCiAgICBlZmZlY3RpdmVfYXBwcm92'
            'ZWQgPSB2b2ljZW92ZXJfYXBwcm92ZWQgYW5kIG5vdCBpc19ndHRzCiAgICBpZiBpc19ndHRzOgogICAgICAgIHN0YXR1cywgc3Rh'
            'dHVzX3JlYXNvbiA9ICJkZXZlbG9wbWVudF9wbGFjZWhvbGRlciIsICJnVFRTIGZyZWUvbG9jYWwgYmFzZWxpbmU7IG5ldmVyIHBy'
            'b2R1Y3Rpb25fcmVhZHkgYnkgcG9saWN5IgogICAgZWxpZiBlZmZlY3RpdmVfYXBwcm92ZWQ6CiAgICAgICAgc3RhdHVzLCBzdGF0'
            'dXNfcmVhc29uID0gImFwcHJvdmVkIiwgImh1bWFuIGFwcHJvdmVkIgogICAgZWxzZToKICAgICAgICBzdGF0dXMsIHN0YXR1c19y'
            'ZWFzb24gPSAicmV2aWV3X3JlcXVpcmVkIiwgImF3YWl0aW5nIGh1bWFuIGFwcHJvdmFsICh2b2ljZW92ZXJfYXBwcm92ZWQ9RmFs'
            'c2UpIgoKICAgIHJldHVybiB7CiAgICAgICAgInN0YXR1cyI6IHN0YXR1cywKICAgICAgICAic3RhdHVzX3JlYXNvbiI6IHN0YXR1'
            'c19yZWFzb24sCiAgICAgICAgInByb3ZpZGVyIjogY2FuZGlkYXRlWyJwcm92aWRlciJdLAogICAgICAgICJtb2RlbCI6IGNhbmRp'
            'ZGF0ZVsibW9kZWwiXSwKICAgICAgICAidm9pY2VfaWQiOiBjYW5kaWRhdGUuZ2V0KCJ2b2ljZV9pZCIpLAogICAgICAgICJhdWRp'
            'b19wYXRoIjogY2FuZGlkYXRlWyJhdWRpb19wYXRoIl0sCiAgICAgICAgInNvdXJjZV9zY3JpcHRfaGFzaCI6IHNvdXJjZV9zY3Jp'
            'cHRfaGFzaCwKICAgICAgICAic291cmNlX3NjcmlwdF9zdGF0dXMiOiBzY3JpcHRfc3RhdHVzLAogICAgICAgICJ1c2VkX2Rldl9m'
            'YWxsYmFja19zY3JpcHQiOiB1c2VkX2Rldl9mYWxsYmFja19zY3JpcHQsCiAgICAgICAgImR1cmF0aW9uX3NlYyI6IGF1ZGlvX2R1'
            'cmF0aW9uX3NlYyhmZnByb2JlX2JpbiwgY2FuZGlkYXRlWyJhdWRpb19wYXRoIl0pLAogICAgICAgICJjb3N0X3VzZF9lc3RpbWF0'
            'ZSI6IGNhbmRpZGF0ZS5nZXQoImNvc3RfdXNkX2VzdGltYXRlIiwgMC4wKSwKICAgICAgICAiY29zdF91c2RfYWN0dWFsIjogTm9u'
            'ZSwKICAgICAgICAibGF0ZW5jeV9zZWMiOiBjYW5kaWRhdGUuZ2V0KCJsYXRlbmN5X3NlYyIsIDAuMCksCiAgICAgICAgInF1YWxp'
            'dHlfc2NvcmVfMV81IjogcXVhbGl0eV9zY29yZV8xXzUsCiAgICAgICAgImxhbmd1YWdlIjogbGFuZ3VhZ2UsCiAgICAgICAgImdl'
            'bmVyYXRlZF9hdCI6IGRhdGV0aW1lLm5vdyh0aW1lem9uZS51dGMpLmlzb2Zvcm1hdCgpLAogICAgICAgICJodW1hbl9hcHByb3Zh'
            'bCI6IHsiYXBwcm92ZWQiOiBlZmZlY3RpdmVfYXBwcm92ZWQsICJub3RlcyI6IGFwcHJvdmFsX25vdGVzIG9yIE5vbmV9LAogICAg'
            'fQoKCkRFVl9GQUxMQkFDS19TQ1JJUFRfVEVYVCA9ICgKICAgICJCdSBzYWhuZWRlIGhpYyBraW1zZW5pbiBiZWtsZW1lZGlnaSBi'
            'aXIgc2V5IG9sdXlvci4gQW5hIGthcmFrdGVyLCBlbiB5YWtpbiBkb3N0dW51biAiCiAgICAiYXNsaW5kYSBiYXNrYSBiaXIgcGxh'
            'bmkgb2xkdWd1bnUgb2dyZW5peW9yLiBQZWtpIGJ1bmRhbiBzb25yYSBuZSBvbGFjYWs/IENldmFiaSBzb24gIgogICAgInNhbml5'
            'ZWRlIGdlbGl5b3IgdmUga2ltc2UgYnVuYSBoYXppciBkZWdpbC4iCikKCgpkZWYgcnVuX3R0cyhzZWxlY3RlZF9zY3JpcHQ6IGRp'
            'Y3QgfCBOb25lLCBvdXRfZGlyLCBmZnByb2JlX2Jpbj1Ob25lLCBlbGV2ZW5sYWJzX2NsaWVudD1Ob25lLAogICAgICAgICAgICBv'
            'cGVuYWlfY2xpZW50PU5vbmUsIGd0dHNfZmFjdG9yeT1Ob25lLCB2b2ljZW92ZXJfYXBwcm92ZWQ6IGJvb2wgPSBGYWxzZSwKICAg'
            'ICAgICAgICAgYXBwcm92YWxfbm90ZXM6IHN0ciA9ICIiKSAtPiBkaWN0OgogICAgIiIiSGlnaC1sZXZlbCwgc2luZ2xlLXJ1biBU'
            'VFMgZW50cnkgcG9pbnQgdXNlZCBieSB0aGUgbWFzdGVyIG5vdGVib29rLgogICAgTWlycm9ycyBub3RlYm9vayAwMydzIGFwcHJv'
            'dmFsIGdhdGUgZXhhY3RseTogcHJvZHVjdGlvbi1xdWFsaXR5IHByb3ZpZGVycwogICAgKEVsZXZlbkxhYnMsIE9wZW5BSSkgb25s'
            'eSBydW4gd2hlbiBzZWxlY3RlZF9zY3JpcHRbJ3N0YXR1cyddID09ICdhcHByb3ZlZCcKICAgIChhIHJlYWwsIGh1bWFuLWFwcHJv'
            'dmVkIHNjcmlwdCk7IG90aGVyd2lzZSBvbmx5IHRoZSBmcmVlIGdUVFMgYmFzZWxpbmUKICAgIHJ1bnMgKG9yLCBpZiBldmVuIHRo'
            'YXQgaGFzIG5vIG5ldHdvcmssIGFuIGluc3VmZmljaWVudF9kYXRhIHJlc3VsdCBpcwogICAgcmV0dXJuZWQpIC0tIHByb2R1Y3Rp'
            'b24gcHJvdmlkZXJzIGFyZSBuZXZlciBzaWxlbnRseSBpbnZva2VkIG9uIGFuCiAgICB1bmFwcHJvdmVkIG9yIG1pc3Npbmcgc2Ny'
            'aXB0LgogICAgIiIiCiAgICBvdXRfZGlyID0gUGF0aChvdXRfZGlyKQogICAgb3V0X2Rpci5ta2RpcihwYXJlbnRzPVRydWUsIGV4'
            'aXN0X29rPVRydWUpCgogICAgdXNpbmdfZGV2X2ZhbGxiYWNrID0gbm90IChzZWxlY3RlZF9zY3JpcHQgYW5kIHNlbGVjdGVkX3Nj'
            'cmlwdC5nZXQoInNjcmlwdF90ZXh0IikpCiAgICBpZiB1c2luZ19kZXZfZmFsbGJhY2s6CiAgICAgICAgc2NyaXB0X3RleHQgPSBE'
            'RVZfRkFMTEJBQ0tfU0NSSVBUX1RFWFQKICAgICAgICBsYW5ndWFnZSA9ICJ0ciIKICAgICAgICBzY3JpcHRfc3RhdHVzID0gImRl'
            'dmVsb3BtZW50X3BsYWNlaG9sZGVyIgogICAgZWxzZToKICAgICAgICBzY3JpcHRfdGV4dCA9IHNlbGVjdGVkX3NjcmlwdFsic2Ny'
            'aXB0X3RleHQiXQogICAgICAgIGxhbmd1YWdlID0gc2VsZWN0ZWRfc2NyaXB0LmdldCgibGFuZ3VhZ2UiLCAidHIiKQogICAgICAg'
            'IHNjcmlwdF9zdGF0dXMgPSBzZWxlY3RlZF9zY3JpcHQuZ2V0KCJzdGF0dXMiLCAicmV2aWV3X3JlcXVpcmVkIikKCiAgICBydW5f'
            'cHJvZHVjdGlvbl90dHMgPSAobm90IHVzaW5nX2Rldl9mYWxsYmFjaykgYW5kIHNjcmlwdF9zdGF0dXMgPT0gImFwcHJvdmVkIgoK'
            'ICAgIGNhbmRpZGF0ZXMgPSBbXQogICAgaWYgcnVuX3Byb2R1Y3Rpb25fdHRzIGFuZCBlbGV2ZW5sYWJzX2NsaWVudCBpcyBub3Qg'
            'Tm9uZToKICAgICAgICBjZmcgPSBUVFNfTU9ERUxfUkVHSVNUUllbImVsZXZlbmxhYnMiXQogICAgICAgIHJlc3VsdCA9IGNhbGxf'
            'ZWxldmVubGFic190dHMoZWxldmVubGFic19jbGllbnQsIHNjcmlwdF90ZXh0LCBvdXRfZGlyIC8gImVsZXZlbmxhYnMubXAzIiwK'
            'ICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBjZmdbIm1vZGVsX2lkIl0sIGNmZ1sidm9pY2VfaWQiXSkKICAg'
            'ICAgICBpZiAiZXJyb3IiIG5vdCBpbiByZXN1bHQ6CiAgICAgICAgICAgIGNhbmRpZGF0ZXMuYXBwZW5kKHJlc3VsdCkKICAgIGlm'
            'IHJ1bl9wcm9kdWN0aW9uX3R0cyBhbmQgb3BlbmFpX2NsaWVudCBpcyBub3QgTm9uZToKICAgICAgICBjZmcgPSBUVFNfTU9ERUxf'
            'UkVHSVNUUllbIm9wZW5haSJdCiAgICAgICAgcmVzdWx0ID0gY2FsbF9vcGVuYWlfdHRzKG9wZW5haV9jbGllbnQsIHNjcmlwdF90'
            'ZXh0LCBvdXRfZGlyIC8gZiJvcGVuYWkte2NmZ1snbW9kZWxfaWQnXX0ubXAzIiwKICAgICAgICAgICAgICAgICAgICAgICAgICAg'
            'ICAgICAgIGNmZ1sibW9kZWxfaWQiXSwgY2ZnWyJ2b2ljZSJdLCBjZmdbInByaWNlX3Blcl8xa19jaGFyc191c2QiXSkKICAgICAg'
            'ICBpZiAiZXJyb3IiIG5vdCBpbiByZXN1bHQ6CiAgICAgICAgICAgIGNhbmRpZGF0ZXMuYXBwZW5kKHJlc3VsdCkKCiAgICBndHRz'
            'X3Jlc3VsdCA9IGNhbGxfZ3R0cyhzY3JpcHRfdGV4dCwgb3V0X2RpciAvICJndHRzLm1wMyIsIGxhbmd1YWdlPWxhbmd1YWdlLCBn'
            'dHRzX2ZhY3Rvcnk9Z3R0c19mYWN0b3J5KQogICAgaWYgImVycm9yIiBub3QgaW4gZ3R0c19yZXN1bHQ6CiAgICAgICAgY2FuZGlk'
            'YXRlcy5hcHBlbmQoZ3R0c19yZXN1bHQpCgogICAgcHJvZHVjdGlvbl9jYW5kaWRhdGVzID0gW2MgZm9yIGMgaW4gY2FuZGlkYXRl'
            'cyBpZiBjWyJwcm92aWRlciJdICE9ICJndHRzIl0KICAgIHNlbGVjdGVkID0gcHJvZHVjdGlvbl9jYW5kaWRhdGVzWzBdIGlmIHBy'
            'b2R1Y3Rpb25fY2FuZGlkYXRlcyBlbHNlIChjYW5kaWRhdGVzWzBdIGlmIGNhbmRpZGF0ZXMgZWxzZSBOb25lKQoKICAgIHJldHVy'
            'biBidWlsZF9zZWxlY3RlZF92b2ljZW92ZXJfYXJ0aWZhY3QoCiAgICAgICAgc2VsZWN0ZWQsIHNjcmlwdF90ZXh0LCBzY3JpcHRf'
            'c3RhdHVzLCB1c2luZ19kZXZfZmFsbGJhY2ssIGZmcHJvYmVfYmluPWZmcHJvYmVfYmluLAogICAgICAgIHZvaWNlb3Zlcl9hcHBy'
            'b3ZlZD12b2ljZW92ZXJfYXBwcm92ZWQsIGFwcHJvdmFsX25vdGVzPWFwcHJvdmFsX25vdGVzLCBsYW5ndWFnZT1sYW5ndWFnZSwK'
            'ICAgICkK'
        ),
    },
}

def _load_local_pipeline_modules():
    # 1) Zaten import edilebiliyorsa (yerel .venv, repo Colab'da klonlanmis) dogrudan kullan.
    try:
        import local_pipeline
        modules = {}
        for _name in _LOCAL_PIPELINE_MODULE_NAMES:
            modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
        print('local_pipeline modulleri dogrudan import edildi:', list(modules))
        return modules
    except Exception:
        pass

    # 2) Drive'da veya proje kokunun bir ustunde daha once birakilmis bir kopya var mi?
    search_roots = []
    try:
        search_roots.append(Path(DRIVE_PROJECT_ROOT))
    except NameError:
        pass
    search_roots += [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for root in search_roots:
        candidate_dir = Path(root) / 'local_pipeline'
        if all((candidate_dir / f'{_n}.py').is_file() for _n in _LOCAL_PIPELINE_MODULE_NAMES):
            sys.path.insert(0, str(candidate_dir.parent))
            try:
                modules = {}
                for _name in _LOCAL_PIPELINE_MODULE_NAMES:
                    modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
                print('local_pipeline modulleri onceden birakilmis kopyadan yuklendi:', candidate_dir)
                return modules
            except Exception:
                pass

    # 3) PROJECT_REPO_URL verilmisse git clone dene.
    if PROJECT_REPO_URL:
        clone_dir = Path('/content/youtube_automation_repo') if os.path.isdir('/content') else Path.cwd() / '_youtube_automation_repo'
        if not (clone_dir / 'local_pipeline' / (_LOCAL_PIPELINE_MODULE_NAMES[0] + '.py')).is_file():
            subprocess.run(['git', 'clone', '--depth', '1', PROJECT_REPO_URL, str(clone_dir)], check=False)
        if all((clone_dir / 'local_pipeline' / f'{_n}.py').is_file() for _n in _LOCAL_PIPELINE_MODULE_NAMES):
            sys.path.insert(0, str(clone_dir))
            try:
                modules = {}
                for _name in _LOCAL_PIPELINE_MODULE_NAMES:
                    modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
                print('local_pipeline modulleri PROJECT_REPO_URL klonundan yuklendi:', clone_dir)
                return modules
            except Exception:
                pass

    # 4) Self-contained fallback: bu hucreye gomulu, build sirasinda
    # local_pipeline/<modul>.py'den birebir uretilmis base64 kopyalari, SHA-256
    # hash'lerini dogrulayarak diske yazip import et.
    fallback_dir = Path(tempfile.mkdtemp(prefix='youtube_automation_bootstrap_'))
    pkg_dir = fallback_dir / 'local_pipeline'
    pkg_dir.mkdir(parents=True, exist_ok=True)
    (pkg_dir / '__init__.py').write_text('', encoding='utf-8')
    for _name, _entry in _LOCAL_PIPELINE_PAYLOADS.items():
        _source_bytes = base64.b64decode(_entry['b64'])
        _actual_digest = hashlib.sha256(_source_bytes).hexdigest()
        if _actual_digest != _entry['sha256']:
            raise RuntimeError(
                f"Gomulu local_pipeline.{_name} payloadi bozuk/tutarsiz: "
                f"beklenen sha256={_entry['sha256']!r}, hesaplanan={_actual_digest!r}. "
                "Bu notebook yeniden uretilmeli (nb_builder ile)."
            )
        (pkg_dir / f'{_name}.py').write_text(_source_bytes.decode('utf-8'), encoding='utf-8')
    sys.path.insert(0, str(fallback_dir))
    modules = {}
    for _name in _LOCAL_PIPELINE_MODULE_NAMES:
        modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
    print('local_pipeline modulleri: repo/Drive/PROJECT_REPO_URL bulunamadi; gomulu (self-contained, hash dogrulanmis) kopyalar kullanildi.')
    return modules

_local_pipeline_modules = _load_local_pipeline_modules()
tts = _local_pipeline_modules['tts']



In [ ]:
# Kanonik TTS model kayit defteri (local_pipeline/tts.py). Resmi katalog
# degisirse tek yerde -- kanonik modulde -- guncellenir, notebooklarda kopyasi yoktur.
TTS_MODEL_REGISTRY = tts.TTS_MODEL_REGISTRY
print('TTS_MODEL_REGISTRY:', json.dumps(TTS_MODEL_REGISTRY, indent=2))


## 1. Notebook 02 çıktısını yükle: `selected_script.json`

Sırayla aranır: `SELECTED_SCRIPT_PATH` (elle) → `results/selected_script.json` → `../results/selected_script.json` → `/content/results/selected_script.json`.

In [ ]:
SELECTED_SCRIPT_PATH = ''  # elle yol vermek istersen buraya yaz

CANDIDATE_SCRIPT_PATHS = [
    SELECTED_SCRIPT_PATH,
    str(RESULTS_DIR / 'selected_script.json'),
    'results/selected_script.json',
    '../results/selected_script.json',
    '/content/results/selected_script.json',
]

def find_first_existing(candidates):
    for candidate in candidates:
        if candidate and Path(candidate).is_file():
            return Path(candidate).resolve()
    return None

selected_script_path = find_first_existing(CANDIDATE_SCRIPT_PATHS)
selected_script = None
if selected_script_path is not None:
    with open(selected_script_path, 'r', encoding='utf-8') as f:
        selected_script = json.load(f)
    print('Yuklendi:', selected_script_path)
    print('Script durumu:', selected_script.get('status'), '-', selected_script.get('status_reason', ''))
else:
    print('UYARI: selected_script.json bulunamadi. Once Notebook 02\'yi calistir. '
          'Bu hucre olmadan asagida SADECE gelistirme (development_placeholder) fallback metni kullanilabilir.')

## 2. Script metnini belirle (gerçek script veya geliştirme fallback)

Gerçek onaylı script yoksa (Notebook 02 hiç çalışmadıysa ya da script_text boşsa), burada **açıkça işaretlenmiş** bir geliştirme metni kullanılır. Bu asla `production_ready` sayılmaz.

In [ ]:
# Kanonik gelistirme fallback metni (local_pipeline/tts.py).
DEV_FALLBACK_SCRIPT_TEXT = tts.DEV_FALLBACK_SCRIPT_TEXT
DEV_FALLBACK_LANGUAGE = 'tr'

USING_DEV_FALLBACK = False
if selected_script is not None and selected_script.get('script_text'):
    SCRIPT_TEXT = selected_script['script_text']
    LANGUAGE = selected_script.get('language', 'tr')
    SCRIPT_STATUS = selected_script.get('status', 'review_required')
    SOURCE_SCRIPT_PROVIDER = selected_script.get('provider')
    SOURCE_SCRIPT_MODEL = selected_script.get('model')
else:
    USING_DEV_FALLBACK = True
    SCRIPT_TEXT = DEV_FALLBACK_SCRIPT_TEXT
    LANGUAGE = DEV_FALLBACK_LANGUAGE
    SCRIPT_STATUS = 'development_placeholder'
    SOURCE_SCRIPT_PROVIDER = None
    SOURCE_SCRIPT_MODEL = None
    print('development_placeholder metin kullaniliyor: gercek onaylanmis script yok.')

SOURCE_SCRIPT_HASH = hashlib.sha256(SCRIPT_TEXT.encode('utf-8')).hexdigest()[:16]
print('Script durumu:', SCRIPT_STATUS, '| dev fallback:', USING_DEV_FALLBACK)
print('Script metni:', SCRIPT_TEXT)

## 3. Onay kapısı — yayın kalitesi TTS sadece onaylı scriptte çalışır

**Kalite politikası:** `selected_script.json` durumu `approved` değilse (veya dosya hiç yoksa), ElevenLabs/OpenAI gibi yayın-kalitesi adaylar ATLANIR; sadece `gTTS` (development_placeholder) çalışır. Bunu geçersiz kılmak istemiyoruz — script'i önce Notebook 02'de onayla.

In [ ]:
audio_files = {}
audio_meta = {}

# Bu notebook'un katkisi: AYNI kanonik call_*() fonksiyonlarini birden fazla
# saglayiciyla yan yana calistirip karsilastirmak.
attempts = []

if RUN_PRODUCTION_TTS and ELEVENLABS_API_KEY:
    from elevenlabs.client import ElevenLabs
    eleven_cfg = TTS_MODEL_REGISTRY['elevenlabs']
    eleven_client = ElevenLabs(api_key=ELEVENLABS_API_KEY)
    name = 'elevenlabs-' + eleven_cfg['model_id'].replace('eleven_', '')
    attempts.append((name, tts.call_elevenlabs_tts(
        eleven_client, SCRIPT_TEXT, TTS_OUT_DIR / 'elevenlabs.mp3',
        eleven_cfg['model_id'], eleven_cfg['voice_id'],
    ), 'review_required'))
elif not RUN_PRODUCTION_TTS:
    print('ElevenLabs atlandi: script onaylanmamis.')
else:
    print('ElevenLabs key yok, atlandi.')

if RUN_PRODUCTION_TTS and OPENAI_API_KEY:
    from openai import OpenAI
    openai_cfg = TTS_MODEL_REGISTRY['openai']
    openai_client = OpenAI(api_key=OPENAI_API_KEY)
    name = f"openai-{openai_cfg['model_id']}"
    attempts.append((name, tts.call_openai_tts(
        openai_client, SCRIPT_TEXT, TTS_OUT_DIR / f"openai-{openai_cfg['model_id']}.mp3",
        openai_cfg['model_id'], openai_cfg['voice'], openai_cfg['price_per_1k_chars_usd'],
    ), 'review_required'))
elif not RUN_PRODUCTION_TTS:
    print('OpenAI TTS atlandi: script onaylanmamis.')
else:
    print('OpenAI key yok, atlandi.')

# gTTS her zaman denenir (anahtar gerektirmez) ama ASLA production sayilmaz.
attempts.append(('gtts-baseline', tts.call_gtts(
    SCRIPT_TEXT, TTS_OUT_DIR / 'gtts.mp3', language=LANGUAGE,
), 'development_placeholder'))

for name, result, ok_status in attempts:
    if 'error' in result:
        log_result('tts', name, None, 0.0, 0.0, result['error'], status='failed')
        print(f'UYARI: {name} basarisiz, atlandi: {result["error"]}')
        continue
    audio_files[name] = Path(result['audio_path'])
    audio_meta[name] = dict(result)
    audio_meta[name]['duration_sec'] = tts.audio_duration_sec(FFPROBE_BIN, result['audio_path'])
    audio_meta[name]['source_script_hash'] = SOURCE_SCRIPT_HASH
    notes = f'{len(SCRIPT_TEXT)} karakter'
    if name == 'gtts-baseline':
        notes = 'ucretsiz baseline, robotik ses beklenir'
    log_result('tts', name, None, result.get('cost_usd_estimate', 0.0),
               result.get('latency_sec', 0.0), notes, status=ok_status)


## 4. Aday TTS Sağlayıcıları

In [ ]:
audio_files = {}
audio_meta = {}

# Bu notebook'un katkisi: AYNI kanonik call_*() fonksiyonlarini birden fazla
# saglayiciyla yan yana calistirip karsilastirmak.
attempts = []

if RUN_PRODUCTION_TTS and ELEVENLABS_API_KEY:
    from elevenlabs.client import ElevenLabs
    eleven_cfg = TTS_MODEL_REGISTRY['elevenlabs']
    eleven_client = ElevenLabs(api_key=ELEVENLABS_API_KEY)
    name = 'elevenlabs-' + eleven_cfg['model_id'].replace('eleven_', '')
    attempts.append((name, tts.call_elevenlabs_tts(
        eleven_client, SCRIPT_TEXT, TTS_OUT_DIR / 'elevenlabs.mp3',
        eleven_cfg['model_id'], eleven_cfg['voice_id'],
    ), 'review_required'))
elif not RUN_PRODUCTION_TTS:
    print('ElevenLabs atlandi: script onaylanmamis.')
else:
    print('ElevenLabs key yok, atlandi.')

if RUN_PRODUCTION_TTS and OPENAI_API_KEY:
    from openai import OpenAI
    openai_cfg = TTS_MODEL_REGISTRY['openai']
    openai_client = OpenAI(api_key=OPENAI_API_KEY)
    name = f"openai-{openai_cfg['model_id']}"
    attempts.append((name, tts.call_openai_tts(
        openai_client, SCRIPT_TEXT, TTS_OUT_DIR / f"openai-{openai_cfg['model_id']}.mp3",
        openai_cfg['model_id'], openai_cfg['voice'], openai_cfg['price_per_1k_chars_usd'],
    ), 'review_required'))
elif not RUN_PRODUCTION_TTS:
    print('OpenAI TTS atlandi: script onaylanmamis.')
else:
    print('OpenAI key yok, atlandi.')

# gTTS her zaman denenir (anahtar gerektirmez) ama ASLA production sayilmaz.
attempts.append(('gtts-baseline', tts.call_gtts(
    SCRIPT_TEXT, TTS_OUT_DIR / 'gtts.mp3', language=LANGUAGE,
), 'development_placeholder'))

for name, result, ok_status in attempts:
    if 'error' in result:
        log_result('tts', name, None, 0.0, 0.0, result['error'], status='failed')
        print(f'UYARI: {name} basarisiz, atlandi: {result["error"]}')
        continue
    audio_files[name] = Path(result['audio_path'])
    audio_meta[name] = dict(result)
    audio_meta[name]['duration_sec'] = tts.audio_duration_sec(FFPROBE_BIN, result['audio_path'])
    audio_meta[name]['source_script_hash'] = SOURCE_SCRIPT_HASH
    notes = f'{len(SCRIPT_TEXT)} karakter'
    if name == 'gtts-baseline':
        notes = 'ucretsiz baseline, robotik ses beklenir'
    log_result('tts', name, None, result.get('cost_usd_estimate', 0.0),
               result.get('latency_sec', 0.0), notes, status=ok_status)


## 6. Final ses seçimi

`SELECTED_TTS_CANDIDATE = None` bırakılırsa en yüksek puanlı yayın-kalitesi aday (gTTS hariç)
otomatik seçilir; hiçbiri yoksa gTTS geliştirme adayı seçilir.

**Onay burada verilmez.** Dışa aktarım yayın-kalitesi adaylar için `review_required`, gTTS için
`development_placeholder` yazar; onay bölüm 8'deki ayrı hücrede, sesi dinledikten sonra verilir.
gTTS **asla** onaylanamaz.

In [ ]:
SELECTED_TTS_CANDIDATE = None  # None = otomatik; ornek: 'elevenlabs-multilingual-v2'

if SELECTED_TTS_CANDIDATE is None:
    production_candidates = [n for n in audio_files if n != 'gtts-baseline']
    scored_production = [
        (n, next((r['quality_score_1_5'] for r in results if r['candidate'] == n), None))
        for n in production_candidates
    ]
    scored_production = [(n, s) for n, s in scored_production if s is not None]
    if scored_production:
        SELECTED_TTS_CANDIDATE = max(scored_production, key=lambda pair: pair[1])[0]
    elif production_candidates:
        SELECTED_TTS_CANDIDATE = production_candidates[0]
    elif 'gtts-baseline' in audio_files:
        SELECTED_TTS_CANDIDATE = 'gtts-baseline'

if SELECTED_TTS_CANDIDATE is None:
    print('Hicbir ses adayi uretilemedi (anahtar yok / hata). '
          'selected_voiceover.json status=insufficient_data olacak.')
else:
    print('Secilen aday:', SELECTED_TTS_CANDIDATE)
    if SELECTED_TTS_CANDIDATE == 'gtts-baseline':
        print('NOT: gTTS politika geregi development_placeholder kalir ve ASLA onaylanamaz.')


## 6. Final ses seçimi ve onayı

`SELECTED_TTS_CANDIDATE = None` bırakılırsa en yüksek `quality_score_1_5` puanlı yayın-kalitesi aday (gTTS hariç) otomatik seçilir; hiçbiri yoksa gTTS geliştirme adayı seçilir. **Bu sesi kullanmadan önce `VOICEOVER_APPROVED = True` yapman gerekir — gTTS asla approved/production_ready olamaz.**

In [ ]:
df = pd.DataFrame(results)
out_path = RESULTS_DIR / '03_tts_voice_sonuclar.csv'
if not df.empty:
    display(df)
    df.to_csv(out_path, index=False)
else:
    pd.DataFrame(columns=['task', 'candidate', 'quality_score_1_5', 'cost_usd_estimate',
                           'latency_sec', 'notes', 'status']).to_csv(out_path, index=False)
print('Kaydedildi:', out_path)

# Kanonik artefakt olusturma (master notebook ile ayni fonksiyon). Bu fonksiyon
# onay argumani ALMAZ: yayin-kalitesi cikti 'review_required', gTTS ciktisi
# 'development_placeholder' olur. Onay bolum 8'deki ayri hucrede verilir.
benchmark_attempts = [
    {'provider': meta.get('provider'), 'model': meta.get('model'), 'ok': True, 'error': None}
    for meta in audio_meta.values()
]

if SELECTED_TTS_CANDIDATE is not None:
    quality_score = next((r['quality_score_1_5'] for r in results
                          if r['candidate'] == SELECTED_TTS_CANDIDATE), None)
    selected_voiceover = tts.build_selected_voiceover_artifact(
        audio_meta[SELECTED_TTS_CANDIDATE], SCRIPT_TEXT, SCRIPT_STATUS, USING_DEV_FALLBACK,
        ffprobe_bin=FFPROBE_BIN, quality_score_1_5=quality_score, language=LANGUAGE,
        attempts=benchmark_attempts,
    )
else:
    selected_voiceover = tts.build_selected_voiceover_artifact(
        None, SCRIPT_TEXT, SCRIPT_STATUS, USING_DEV_FALLBACK, ffprobe_bin=FFPROBE_BIN,
        language=LANGUAGE, attempts=benchmark_attempts,
    )

selected_voiceover_path = RESULTS_DIR / 'selected_voiceover.json'
with open(selected_voiceover_path, 'w', encoding='utf-8') as f:
    json.dump(selected_voiceover, f, ensure_ascii=False, indent=2)
print('Kaydedildi:', selected_voiceover_path, '(status:', selected_voiceover['status'], ')')

zip_path = RESULTS_DIR / '03_tts_voice_paylasim_paketi.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(selected_voiceover_path, arcname='selected_voiceover.json')
    if out_path.exists():
        archive.write(out_path, arcname=out_path.name)
    for audio_path in audio_files.values():
        archive.write(audio_path, arcname=f'tts/{Path(audio_path).name}')
print('Paylasim paketi hazir:', zip_path)


## 8. İnsan onayı (ayrı adım, üretimden sonra)

Sesi **dinledikten sonra** inceleme çıktısının yazdırdığı SHA-256'yı `VOICEOVER_REVIEWED_SHA256`
alanına yapıştır. Ses dosyası sonradan değişirse onay otomatik geçersizleşir.

In [ ]:
if selected_voiceover.get('audio_path'):
    voiceover_review_info = approval.describe_for_review('voiceover', selected_voiceover)
    if voiceover_review_info['reviewable']:
        print('Dinledigin ses dosyasinin SHA-256 degeri (asagiya yapistir):')
        print('   ', voiceover_review_info['content_sha256'])
    else:
        print('ONAYLANAMAZ:', voiceover_review_info['blocker'])
else:
    print('Onaylanacak bir seslendirme yok.')

VOICEOVER_REVIEWED_SHA256 = ''   # yukaridaki SHA-256'yi buraya yapistir
VOICEOVER_APPROVED_BY = ''
VOICEOVER_APPROVAL_NOTES = ''

if not selected_voiceover.get('audio_path'):
    pass
elif not VOICEOVER_REVIEWED_SHA256 or not VOICEOVER_APPROVED_BY:
    print('Onay ATLANDI: VOICEOVER_REVIEWED_SHA256 ve VOICEOVER_APPROVED_BY doldurulmali.')
    print('status olarak kaliyor:', selected_voiceover['status'])
else:
    try:
        selected_voiceover = approval.approve_artifact(
            selected_voiceover, 'voiceover', VOICEOVER_REVIEWED_SHA256,
            VOICEOVER_APPROVED_BY, VOICEOVER_APPROVAL_NOTES,
        )
        with open(selected_voiceover_path, 'w', encoding='utf-8') as f:
            json.dump(selected_voiceover, f, ensure_ascii=False, indent=2)
        print('Seslendirme ONAYLANDI ->', selected_voiceover_path,
              '(status:', selected_voiceover['status'], ')')
    except ValueError as exc:
        print('ONAY REDDEDILDI:', exc)
        print('status olarak kaliyor:', selected_voiceover['status'])


## 7. Dışa aktarım